# TMS Algorithm Benchmark — Google Colab

Notebook này chạy bộ kiểm thử và benchmark thuật toán TMS trực tiếp từ GitHub, hiển thị log theo thời gian thực và lưu đầy đủ bằng chứng của từng lần chạy.

Kết quả cuối cùng gồm:

- `run.log` tổng và log riêng cho từng bước;
- metadata phần cứng, phiên bản thư viện, commit Git và cấu hình benchmark;
- báo cáo gốc Markdown;
- bảng so sánh chi tiết/tổng hợp ở định dạng CSV và JSON;
- biểu đồ PNG về thời gian, tỷ lệ phục vụ và chi phí hợp lệ;
- đúng một file ZIP được lưu vào Google Drive để tải về và gửi phân tích.

> **Lưu ý hiệu năng:** OR-Tools và các metaheuristic hiện tại dùng CPU. Bật GPU trên Colab không tự làm chúng nhanh hơn; ưu tiên runtime có nhiều CPU/RAM. Chỉ các kết quả vượt qua validator mới được dùng để so sánh chi phí.

Sau khi đã push mã nguồn lên GitHub, chỉnh cấu hình ở cell đầu tiên nếu cần rồi chọn **Runtime → Run all**.


## 1. Cấu hình lần chạy

In [ ]:
CONFIG = {
    # Repository phải chứa optimizer và bộ benchmark production.
    "repo_url": "https://github.com/Bang334/Project-Logictists.git",
    "repo_ref": "feat/phase-1-core-tms",

    # quick: kiểm tra nhanh; standard: so sánh chính; deep: chạy lâu để tìm nghiệm tốt hơn.
    "profile": "standard",
    "workers": None,  # None = tự chọn theo số CPU, tối đa 4 process song song.

    "run_tests": True,
    "run_parallel_benchmark": True,
    "run_single_dataset": False,
    "single_dataset": "optimizer/benchmarks/cases/db_hanoi_16_orders.json",
    "single_seeds": "0,1,2",
    "run_scalability_benchmark": True,
    "scalability_sizes": [50, 100, 200],
    # Mỗi lần chạy chỉ xuất một ZIP vào thư mục Drive này.
    "save_zip_to_drive": True,
    "drive_folder": "MyDrive/TMS_Benchmark_Results",
    "auto_download_zip": False,  # Bật nếu muốn tải thêm một bản qua trình duyệt.
}

PROFILES = {
    "quick": {"search_budget": 2.0, "spatial_audit_budget": 5.0, "repair_budget": 8.0,
              "single_time_limit": 2.0, "scalability_time_limit": 3},
    "standard": {"search_budget": 10.0, "spatial_audit_budget": 15.0, "repair_budget": 20.0,
                 "single_time_limit": 10.0, "scalability_time_limit": 5},
    "deep": {"search_budget": 60.0, "spatial_audit_budget": 30.0, "repair_budget": 45.0,
             "single_time_limit": 60.0, "scalability_time_limit": 15},
}

if CONFIG["profile"] not in PROFILES:
    raise ValueError(f"profile phải thuộc {sorted(PROFILES)}, nhận được: {CONFIG['profile']!r}")

PROFILE = PROFILES[CONFIG["profile"]]
print("Cấu hình đã chọn:", CONFIG)
print("Ngân sách chạy:", PROFILE)


## 2. Khởi tạo thư mục, logger và ghi nhận môi trường

In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import os
import platform
import queue
import shlex
import shutil
import subprocess
import sys
import threading
import time
from collections import deque
from datetime import datetime, timezone
from pathlib import Path


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
WORKSPACE_DIR = Path("/content/tms_workspace") / RUN_ID
PROJECT_DIR = WORKSPACE_DIR / "project"
RUN_DIR = Path("/content/tms_results") / RUN_ID
LOG_DIR = RUN_DIR / "logs"
REPORT_DIR = RUN_DIR / "reports"
PLOT_DIR = RUN_DIR / "plots"
for directory in (WORKSPACE_DIR, RUN_DIR, LOG_DIR, REPORT_DIR, PLOT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MASTER_LOG = RUN_DIR / "run.log"
RUN_RECORDS = []


def log(message: str, level: str = "INFO") -> None:
    line = f"[{utc_now()}] [{level}] {message}"
    print(line, flush=True)
    with MASTER_LOG.open("a", encoding="utf-8") as handle:
        handle.write(line + "\n")


def write_json(path: Path, value) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding="utf-8")


def run_logged(
    name: str,
    command: list[str],
    *,
    cwd: Path | None = None,
    check: bool = False,
    capture_output: bool = False,
) -> dict:
    """Chạy lệnh, stream stdout/stderr ra cell và đồng thời lưu log có timestamp."""
    step_log = LOG_DIR / f"{name}.log"
    safe_command = shlex.join(str(part) for part in command)
    started_at = time.perf_counter()
    started_utc = utc_now()
    log(f"BẮT ĐẦU {name}: {safe_command}")
    tail = deque(maxlen=80)
    captured = [] if capture_output else None
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    if PROJECT_DIR.exists():
        existing = env.get("PYTHONPATH", "")
        env["PYTHONPATH"] = str(PROJECT_DIR) + (os.pathsep + existing if existing else "")

    with step_log.open("w", encoding="utf-8") as handle:
        handle.write(f"started_utc={started_utc}\n")
        handle.write(f"cwd={cwd or Path.cwd()}\n")
        handle.write(f"command={safe_command}\n\n")
        process = subprocess.Popen(
            [str(part) for part in command],
            cwd=str(cwd) if cwd else None,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        )
        assert process.stdout is not None
        output_queue = queue.Queue()

        def read_output() -> None:
            for raw_line in process.stdout:
                output_queue.put(raw_line)
            output_queue.put(None)

        reader = threading.Thread(target=read_output, daemon=True)
        reader.start()
        while True:
            try:
                raw_line = output_queue.get(timeout=15)
            except queue.Empty:
                elapsed = time.perf_counter() - started_at
                heartbeat = f"[{utc_now()}] [HEARTBEAT] {name} vẫn đang chạy ({elapsed:.0f}s)"
                print(heartbeat, flush=True)
                handle.write(heartbeat + "\n")
                handle.flush()
                with MASTER_LOG.open("a", encoding="utf-8") as master:
                    master.write(heartbeat + "\n")
                continue
            if raw_line is None:
                break
            clean = raw_line.rstrip("\n")
            stamped = f"[{utc_now()}] {clean}"
            print(stamped, flush=True)
            handle.write(stamped + "\n")
            handle.flush()
            with MASTER_LOG.open("a", encoding="utf-8") as master:
                master.write(stamped + "\n")
            tail.append(clean)
            if captured is not None:
                captured.append(raw_line)
        reader.join()
        return_code = process.wait()

    duration = round(time.perf_counter() - started_at, 3)
    status = "PASS" if return_code == 0 else "FAIL"
    record = {
        "name": name,
        "status": status,
        "return_code": return_code,
        "started_utc": started_utc,
        "finished_utc": utc_now(),
        "duration_seconds": duration,
        "command": safe_command,
        "cwd": str(cwd or Path.cwd()),
        "log_file": str(step_log.relative_to(RUN_DIR)),
        "tail": list(tail),
    }
    if captured is not None:
        record["stdout"] = "".join(captured)
    RUN_RECORDS.append(record)
    log(f"KẾT THÚC {name}: {status}, exit={return_code}, thời gian={duration:.3f}s",
        "INFO" if return_code == 0 else "ERROR")
    write_json(RUN_DIR / "run_status.json", RUN_RECORDS)
    if check and return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    return record


def detect_gpu() -> dict:
    command = [
        "nvidia-smi",
        "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader,nounits",
    ]
    try:
        result = subprocess.run(command, capture_output=True, text=True, check=True, timeout=10)
        rows = [line.strip() for line in result.stdout.splitlines() if line.strip()]
        return {"available": bool(rows), "devices": rows}
    except (FileNotFoundError, subprocess.SubprocessError):
        return {"available": False, "devices": []}


def memory_gib() -> float | None:
    try:
        values = Path("/proc/meminfo").read_text(encoding="utf-8").splitlines()
        total_kib = int(next(line for line in values if line.startswith("MemTotal:" )).split()[1])
        return round(total_kib / 1024 / 1024, 2)
    except (OSError, StopIteration, ValueError):
        return None


gpu = detect_gpu()
ENVIRONMENT = {
    "run_id": RUN_ID,
    "started_utc": utc_now(),
    "platform": platform.platform(),
    "python": platform.python_version(),
    "cpu_logical_count": os.cpu_count(),
    "memory_gib": memory_gib(),
    "gpu": gpu,
    "gpu_note": "GPU chỉ được ghi nhận; solver hiện tại dùng CPU và không tự tăng tốc bằng GPU.",
    "config": CONFIG,
    "profile_values": PROFILE,
}
write_json(RUN_DIR / "environment.json", ENVIRONMENT)
log(f"Run ID: {RUN_ID}")
log(f"CPU logic: {os.cpu_count()} | RAM: {ENVIRONMENT['memory_gib']} GiB | GPU: {gpu['devices'] or 'không có'}")
log("Solver hiện tại chạy CPU; GPU không được dùng để tuyên bố tăng tốc.")
print(f"\nThư mục kết quả: {RUN_DIR}")


## 3. Clone đúng phiên bản mã nguồn từ GitHub

In [ ]:
if (PROJECT_DIR / ".git").is_dir():
    log(f"Tái sử dụng source đã clone trong runtime hiện tại: {PROJECT_DIR}")
else:
    clone_record = run_logged(
        "git_clone",
        [
            "git", "clone", "--depth", "1", "--branch", CONFIG["repo_ref"],
            CONFIG["repo_url"], str(PROJECT_DIR),
        ],
        cwd=WORKSPACE_DIR,
        check=True,
    )

required_paths = [
    PROJECT_DIR / "optimizer" / "benchmark_hybrid_regression.py",
    PROJECT_DIR / "optimizer" / "benchmarks" / "cases",
    PROJECT_DIR / "optimizer" / "requirements.txt",
]
missing = [str(path.relative_to(PROJECT_DIR)) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Repository chưa chứa đủ optimizer và benchmark production. Hãy push thay đổi rồi chạy lại Runtime: "
        + ", ".join(missing)
    )

commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR,
    capture_output=True, text=True, check=True,
).stdout.strip()
ENVIRONMENT["source"] = {
    "repository": CONFIG["repo_url"],
    "ref": CONFIG["repo_ref"],
    "commit": commit,
}
write_json(RUN_DIR / "environment.json", ENVIRONMENT)
log(f"Đã checkout commit {commit}")

for path in (PROJECT_DIR, PROJECT_DIR / "optimizer"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
os.chdir(PROJECT_DIR)
print(f"Project root: {PROJECT_DIR}")


## 4. Cài dependencies và ghi phiên bản

In [ ]:
dependency_record = run_logged(
    "install_dependencies",
    [
        sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
        "-r", str(PROJECT_DIR / "optimizer" / "requirements.txt"),
        "pandas", "matplotlib", "psutil", "tabulate",
    ],
    cwd=PROJECT_DIR,
    check=True,
)

tracked_packages = [
    "ortools", "numpy", "pydantic", "fastapi", "pytest",
    "pandas", "matplotlib", "psutil",
]
package_versions = {}
for package in tracked_packages:
    try:
        package_versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        package_versions[package] = None
ENVIRONMENT["packages"] = package_versions
write_json(RUN_DIR / "environment.json", ENVIRONMENT)
log("Phiên bản thư viện: " + json.dumps(package_versions, ensure_ascii=False))


## 5. Kiểm tra module và manifest dữ liệu

Cell này dừng sớm nếu mã nguồn vừa clone không đúng contract notebook cần dùng.


In [ ]:
benchmark_manifest = json.loads((PROJECT_DIR / "optimizer" / "benchmarks" / "best_known_results.json").read_text(encoding="utf-8"))
ALGORITHM_NAMES = ["Production Hybrid ALNS"]
DATASETS = [(row["dataset"], row["file"]) for row in benchmark_manifest["cases"]]
from optimizer.app.spatial_validator import SpatialValidator

dataset_manifest = []
for label, filename in DATASETS:
    path = PROJECT_DIR / "optimizer" / "benchmarks" / "cases" / filename
    dataset_manifest.append({
        "label": label,
        "filename": filename,
        "exists": path.is_file(),
        "size_bytes": path.stat().st_size if path.is_file() else None,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest() if path.is_file() else None,
    })

missing_datasets = [row["filename"] for row in dataset_manifest if not row["exists"]]
if missing_datasets:
    raise FileNotFoundError("Thiếu dataset: " + ", ".join(missing_datasets))

write_json(RUN_DIR / "dataset_manifest.json", dataset_manifest)
log(f"Đã xác minh {len(ALGORITHM_NAMES)} thuật toán và {len(DATASETS)} dataset.")
print("Thuật toán:", ", ".join(ALGORITHM_NAMES))
print("Dataset:")
for row in dataset_manifest:
    print(f"- {row['label']}: {row['filename']} ({row['size_bytes']:,} bytes)")


## 6. Chạy test

Test lỗi vẫn được lưu đầy đủ và notebook tiếp tục benchmark để không làm mất bằng chứng của các phần độc lập.


In [ ]:
if CONFIG["run_tests"]:
    test_record = run_logged(
        "pytest",
        [sys.executable, "-m", "pytest", "-q", "optimizer/tests"],
        cwd=PROJECT_DIR,
        check=False,
    )
    if test_record["return_code"] != 0:
        log("Có test FAIL. Benchmark vẫn tiếp tục và kết quả cuối sẽ được đánh dấu có lỗi.", "WARNING")
else:
    log("Bỏ qua pytest theo cấu hình.", "WARNING")


## 7. Benchmark so sánh 6 thuật toán × 12 dataset

Mỗi nghiệm được audit bằng cùng contract, time-window và production `SpatialValidator`. Các process chạy theo dataset; số worker được giới hạn để tránh làm nghẽn Colab.


In [ ]:
cpu_count = os.cpu_count() or 2
workers = CONFIG["workers"] if CONFIG["workers"] is not None else min(4, cpu_count)
workers = max(1, min(int(workers), len(DATASETS)))
parallel_output_dir = REPORT_DIR / "parallel"
parallel_output_dir.mkdir(parents=True, exist_ok=True)

if CONFIG["run_parallel_benchmark"]:
    benchmark_record = run_logged(
        "parallel_benchmark",
        [
            sys.executable, "optimizer/benchmark_hybrid_regression.py",
            "--time-limit", str(PROFILE["search_budget"]),
            "--seeds", "0", "1", "2",
            "--json-output", str(parallel_output_dir / "hybrid_results.json"),
        ],
        cwd=PROJECT_DIR,
        check=False,
    )
    if benchmark_record["return_code"] != 0:
        log("Benchmark song song lỗi; xem logs/parallel_benchmark.log.", "ERROR")
else:
    log("Bỏ qua benchmark song song theo cấu hình.", "WARNING")


## 8. Benchmark một dataset với nhiều seed (tùy chọn)

In [ ]:
if CONFIG["run_single_dataset"]:
    single_dataset = PROJECT_DIR / CONFIG["single_dataset"]
    if not single_dataset.is_file():
        raise FileNotFoundError(f"Không tìm thấy dataset: {single_dataset}")
    run_logged(
        "single_dataset_benchmark",
        [
            sys.executable, "optimizer/benchmark_hybrid_regression.py",
            "--cases", single_dataset.name,
            "--time-limit", str(PROFILE["single_time_limit"]),
            "--seeds", *CONFIG["single_seeds"].split(","),
            "--json-output", str(REPORT_DIR / "single_dataset_result.json"),
        ],
        cwd=PROJECT_DIR,
        check=False,
    )
else:
    log("Bỏ qua benchmark một dataset theo cấu hình.")


## 9. Benchmark khả năng mở rộng của production solver

Đây là dữ liệu tổng hợp xác định, không phải khoảng cách đường bộ thật. JSON gốc được giữ nguyên để đối chiếu.


In [ ]:
if CONFIG["run_scalability_benchmark"]:
    scalability_record = run_logged(
        "scalability_benchmark",
        [
            sys.executable, "optimizer/benchmark.py",
            "--sizes", *[str(size) for size in CONFIG["scalability_sizes"]],
            "--time-limit", str(PROFILE["scalability_time_limit"]),
            "--wall-timeout", str(max(30, PROFILE["scalability_time_limit"] * 3)),
        ],
        cwd=PROJECT_DIR,
        check=False,
        capture_output=True,
    )
    raw_scalability = scalability_record.pop("stdout", "")
    write_json(RUN_DIR / "run_status.json", RUN_RECORDS)
    (REPORT_DIR / "scalability_raw.json").write_text(raw_scalability, encoding="utf-8")
    try:
        parsed_scalability = json.loads(raw_scalability)
        write_json(REPORT_DIR / "scalability.json", parsed_scalability)
    except json.JSONDecodeError as exc:
        log(f"Không parse được JSON scalability: {exc}", "ERROR")
else:
    log("Bỏ qua scalability benchmark theo cấu hình.")


## 10. Xuất CSV/JSON, tạo biểu đồ và bảng tổng kết

In [ ]:
import math
import re

import matplotlib.pyplot as plt
import pandas as pd


def parse_int(text: str):
    digits = re.sub(r"[^0-9-]", "", text)
    return int(digits) if digits and digits != "-" else None


def parse_parallel_markdown(path: Path) -> pd.DataFrame:
    lines = path.read_text(encoding="utf-8").splitlines()
    in_detail = False
    rows = []
    for line in lines:
        if line.strip() == "## Kết quả chi tiết":
            in_detail = True
            continue
        if in_detail and line.startswith("## "):
            break
        if not in_detail or not line.startswith("|"):
            continue
        cells = [cell.strip() for cell in line.strip().strip("|").split("|")]
        if len(cells) != 9 or cells[0] in {"Tệp dữ liệu", "---"} or cells[0].startswith("---"):
            continue
        served_match = re.match(r"(\d+)/(\d+)\s+\(([0-9.]+)%\)", cells[5])
        if not served_match:
            raise ValueError(f"Không đọc được cột Giao: {cells[5]!r}")
        audit = cells[8]
        rows.append({
            "dataset_file": cells[0].strip("`"),
            "dataset": cells[1],
            "algorithm": cells[2],
            "time_sec": float(cells[3].rstrip("s")),
            "vehicles": int(cells[4]),
            "served": int(served_match.group(1)),
            "orders": int(served_match.group(2)),
            "fulfillment_percent": float(served_match.group(3)),
            "cost_vnd": parse_int(cells[6]) if audit == "PASS" else None,
            "distance_km": float(cells[7]),
            "audit": audit,
        })
    if not rows:
        raise ValueError(f"Không tìm thấy dòng kết quả chi tiết trong {path}")
    return pd.DataFrame(rows)


def build_summary(detail: pd.DataFrame) -> pd.DataFrame:
    dataset_count = detail["dataset_file"].nunique()
    summaries = []
    for algorithm, group in detail.groupby("algorithm", sort=False):
        valid = group[group["audit"] == "PASS"]
        pass_count = len(valid)
        orders = int(group["orders"].sum())
        served = int(valid["served"].sum())
        all_pass = pass_count == dataset_count
        summaries.append({
            "algorithm": algorithm,
            "pass_count": pass_count,
            "dataset_count": dataset_count,
            "validated_served": served,
            "orders": orders,
            "validated_fulfillment_percent": round(served / max(1, orders) * 100, 2),
            "comparable_total_cost_vnd": int(valid["cost_vnd"].sum()) if all_pass else None,
            "total_time_sec": round(float(group["time_sec"].sum()), 3),
            "all_datasets_pass": all_pass,
        })
    summary = pd.DataFrame(summaries)
    summary["cost_sort"] = summary["comparable_total_cost_vnd"].fillna(float("inf"))
    summary = summary.sort_values(
        ["pass_count", "validated_served", "cost_sort", "total_time_sec"],
        ascending=[False, False, True, True],
    ).drop(columns=["cost_sort"]).reset_index(drop=True)
    summary.insert(0, "rank", range(1, len(summary) + 1))
    return summary


report_candidates = sorted(parallel_output_dir.glob("*.md"))
DETAIL = None
SUMMARY = None
if report_candidates:
    source_report = report_candidates[-1]
    DETAIL = parse_parallel_markdown(source_report)
    SUMMARY = build_summary(DETAIL)
    DETAIL.to_csv(REPORT_DIR / "comparison_detail.csv", index=False, encoding="utf-8-sig")
    DETAIL.to_json(REPORT_DIR / "comparison_detail.json", orient="records", force_ascii=False, indent=2)
    SUMMARY.to_csv(REPORT_DIR / "comparison_summary.csv", index=False, encoding="utf-8-sig")
    SUMMARY.to_json(REPORT_DIR / "comparison_summary.json", orient="records", force_ascii=False, indent=2)

    print("BẢNG KẾT QUẢ CHI TIẾT (chỉ chi phí PASS mới được giữ để so sánh):")
    display(DETAIL)
    print("BẢNG XẾP HẠNG TỔNG HỢP:")
    display(SUMMARY)

    runtime = DETAIL.groupby("algorithm", sort=False)["time_sec"].sum().sort_values()
    ax = runtime.plot(kind="barh", figsize=(10, 5), color="#1677ff")
    ax.set_title("Tổng thời gian theo thuật toán")
    ax.set_xlabel("Giây")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "runtime_by_algorithm.png", dpi=160)
    plt.show()

    fulfillment = DETAIL.assign(
        validated_fulfillment=DETAIL["fulfillment_percent"].where(DETAIL["audit"] == "PASS", 0)
    ).groupby("algorithm", sort=False)["validated_fulfillment"].mean().sort_values()
    ax = fulfillment.plot(kind="barh", figsize=(10, 5), color="#52c41a")
    ax.set_title("Tỷ lệ phục vụ trung bình đã qua validator")
    ax.set_xlabel("Phần trăm")
    ax.set_xlim(0, 100)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "validated_fulfillment.png", dpi=160)
    plt.show()

    valid_cost = DETAIL[(DETAIL["audit"] == "PASS") & DETAIL["cost_vnd"].notna()]
    if not valid_cost.empty:
        cost = valid_cost.groupby("algorithm", sort=False)["cost_vnd"].sum().sort_values()
        ax = (cost / 1_000_000).plot(kind="barh", figsize=(10, 5), color="#fa8c16")
        ax.set_title("Tổng chi phí của các lượt PASS")
        ax.set_xlabel("Triệu VND")
        plt.tight_layout()
        plt.savefig(PLOT_DIR / "validated_cost.png", dpi=160)
        plt.show()

    log(f"Đã xuất {len(DETAIL)} dòng chi tiết và {len(SUMMARY)} dòng tổng hợp.")
else:
    log("Không có báo cáo Markdown của benchmark song song để chuyển đổi.", "ERROR")


## 11. Đóng gói và tải toàn bộ kết quả

In [ ]:
# Loại stdout lớn khỏi status và ghi trạng thái cuối cùng.
for record in RUN_RECORDS:
    record.pop("stdout", None)

failed_steps = [record for record in RUN_RECORDS if record["status"] != "PASS"]
FINAL_STATUS = "COMPLETED" if not failed_steps else "COMPLETED_WITH_ERRORS"

summary_lines = [
    "# TMS Colab benchmark run",
    "",
    f"- Run ID: `{RUN_ID}`",
    f"- Trạng thái: **{FINAL_STATUS}**",
    f"- Commit: `{ENVIRONMENT.get('source', {}).get('commit', 'unknown')}`",
    f"- Profile: `{CONFIG['profile']}`",
    f"- Workers: `{workers}`",
    f"- Drive output: {CONFIG['drive_folder']}/tms_benchmark_{RUN_ID}.zip",
    f"- Bắt đầu (UTC): `{ENVIRONMENT['started_utc']}`",
    f"- Kết thúc (UTC): `{utc_now()}`",
    "",
    "## Các bước",
    "",
    "| Bước | Trạng thái | Exit code | Thời gian | Log |",
    "|---|:---:|---:|---:|---|",
]
for record in RUN_RECORDS:
    summary_lines.append(
        f"| {record['name']} | {record['status']} | {record['return_code']} | "
        f"{record['duration_seconds']:.3f}s | `{record['log_file']}` |"
    )

if SUMMARY is not None and not SUMMARY.empty:
    winner = SUMMARY.iloc[0]
    summary_lines.extend([
        "",
        "## Xếp hạng best-found",
        "",
        f"Đứng đầu theo tiêu chí validator → số đơn phục vụ → chi phí so sánh → thời gian: "
        f"**{winner['algorithm']}**.",
        "",
        "> Đây là kết quả best-found trong ngân sách chạy, không phải bằng chứng tối ưu toàn cục.",
        "",
        SUMMARY.to_markdown(index=False),
    ])

(RUN_DIR / "SUMMARY.md").write_text("\n".join(summary_lines) + "\n", encoding="utf-8")
ENVIRONMENT["finished_utc"] = utc_now()
ENVIRONMENT["final_status"] = FINAL_STATUS
write_json(RUN_DIR / "environment.json", ENVIRONMENT)
write_json(RUN_DIR / "run_status.json", RUN_RECORDS)
log(f"TRẠNG THÁI CUỐI: {FINAL_STATUS}")
log(f"Báo cáo tóm tắt: {RUN_DIR / 'SUMMARY.md'}")
if failed_steps:
    log("Các bước lỗi: " + ", ".join(record["name"] for record in failed_steps), "ERROR")
else:
    log("Tất cả bước đã bật trong cấu hình đều PASS.")

zip_filename = f"tms_benchmark_{RUN_ID}.zip"
drive_export_dir = None
if CONFIG["save_zip_to_drive"]:
    drive_folder = Path(CONFIG["drive_folder"])
    if drive_folder.is_absolute() or ".." in drive_folder.parts:
        raise ValueError("drive_folder phải là đường dẫn tương đối an toàn, ví dụ MyDrive/TMS_Benchmark_Results")
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError("Chỉ có thể lưu vào Google Drive khi chạy trong Google Colab") from exc
    log("Đang mount Google Drive; Colab có thể yêu cầu bạn xác nhận quyền truy cập.")
    drive.mount("/content/drive", force_remount=False)
    drive_export_dir = Path("/content/drive") / drive_folder
    drive_export_dir.mkdir(parents=True, exist_ok=True)
    log(f"Google Drive đã sẵn sàng. ZIP sẽ được lưu tại: {drive_export_dir / zip_filename}")

# Manifest được tạo sau cùng để ghi checksum cho các artifact (trừ chính manifest và ZIP).
artifacts = []
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file() and path.name != "artifact_manifest.json":
        artifacts.append({
            "path": str(path.relative_to(RUN_DIR)),
            "size_bytes": path.stat().st_size,
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        })
write_json(RUN_DIR / "artifact_manifest.json", artifacts)

zip_base = Path("/content") / zip_filename[:-4]
zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=RUN_DIR))
drive_zip_path = None
if drive_export_dir is not None:
    drive_zip_path = drive_export_dir / zip_filename
    shutil.copy2(zip_path, drive_zip_path)
    local_hash = hashlib.sha256(zip_path.read_bytes()).hexdigest()
    drive_hash = hashlib.sha256(drive_zip_path.read_bytes()).hexdigest()
    if local_hash != drive_hash or zip_path.stat().st_size != drive_zip_path.stat().st_size:
        raise IOError("ZIP trên Google Drive không khớp với bản local sau khi copy")

print("\n" + "=" * 88)
print("KẾT THÚC TMS BENCHMARK")
print("=" * 88)
print(f"Trạng thái       : {FINAL_STATUS}")
print(f"Run ID           : {RUN_ID}")
print(f"Commit           : {ENVIRONMENT.get('source', {}).get('commit', 'unknown')}")
print(f"Thư mục kết quả  : {RUN_DIR}")
print(f"Log tổng         : {MASTER_LOG}")
print(f"Báo cáo tóm tắt  : {RUN_DIR / 'SUMMARY.md'}")
print(f"ZIP local         : {zip_path} ({zip_path.stat().st_size / 1024:.1f} KiB)")
if drive_zip_path is not None:
    print(f"ZIP trên Drive    : {drive_zip_path}")
    print(f"SHA-256           : {local_hash}")
if failed_steps:
    print("Các bước lỗi      : " + ", ".join(record["name"] for record in failed_steps))
    print("Hãy xem log riêng trong thư mục logs/; kết quả thành công khác vẫn được giữ lại.")
else:
    print("Tất cả bước đã bật trong cấu hình đều PASS.")
print("=" * 88)

if CONFIG["auto_download_zip"]:
    try:
        from google.colab import files
        files.download(str(zip_path))
    except ImportError:
        print("Không ở Google Colab; bỏ qua tải tự động.")
